CONNECT GOOGLE DRIVE

In [1]:
import os
import numpy as np
import pickle
import matplotlib.pyplot as plt
from google.colab import drive

# -----------------------------------------------------------
# CELL 1: SETUP & PATHS
# -----------------------------------------------------------
print("--- CELL 1: SETUP ---")

# 1. Mount Drive
if os.path.ismount('/content/drive'):
    print("✓ Drive already mounted.")
else:
    drive.mount('/content/drive', force_remount=True)

# 2. Define Paths
BASE_DIR = "/content/drive/MyDrive/FabricDefect"
FEATURE_DIR = f"{BASE_DIR}/features"

# 3. Verify Files from Notebook 05 exist
required = ['features_train.npy', 'labels_train.npy', 'features_test.npy', 'labels_test.npy']
missing = [f for f in required if not os.path.exists(os.path.join(FEATURE_DIR, f))]

if missing:
    raise FileNotFoundError(f"Missing files from Notebook 05: {missing}. Please run the previous notebook first.")
else:
    print("✓ All feature files found. Ready to process.")

--- CELL 1: SETUP ---
Mounted at /content/drive
✓ All feature files found. Ready to process.


In [2]:
# -----------------------------------------------------------
# CELL 2: LOAD RAW ARRAYS
# -----------------------------------------------------------
print("\n--- CELL 2: LOADING RAW DATA ---")
print("Loading raw features from disk...")

X_train_raw = np.load(f"{FEATURE_DIR}/features_train.npy")
y_train_raw = np.load(f"{FEATURE_DIR}/labels_train.npy")
X_test_raw  = np.load(f"{FEATURE_DIR}/features_test.npy")
y_test_raw  = np.load(f"{FEATURE_DIR}/labels_test.npy")

print(f"Raw Train Shape: {X_train_raw.shape}")
print(f"Raw Test Shape:  {X_test_raw.shape}")
print(f"Sample Label: {y_train_raw[0]} (Type: {type(y_train_raw[0])})")



--- CELL 2: LOADING RAW DATA ---
Loading raw features from disk...
Raw Train Shape: (2191, 2048)
Raw Test Shape:  (548, 2048)
Sample Label: 1 (Type: <class 'numpy.int64'>)


In [3]:
# -----------------------------------------------------------
# CELL 3: MANUAL LABEL ENCODER (NO SKLEARN)
# -----------------------------------------------------------
print("\n--- CELL 3: MANUAL LABEL ENCODING ---")

def manual_label_encoder(labels):
    """
    Converts string labels to integers manually.
    Returns encoded labels and the mapping dictionary.
    """
    # 1. Find unique classes and sort them (to ensure deterministic order)
    unique_classes = np.unique(labels)
    unique_classes.sort()

    # 2. Create Mappings
    cls_to_idx = {cls_name: i for i, cls_name in enumerate(unique_classes)}
    idx_to_cls = {i: cls_name for i, cls_name in enumerate(unique_classes)}

    # 3. Vectorized Mapping using list comprehension
    # Handle if labels are already integers
    if np.issubdtype(labels.dtype, np.integer) or np.issubdtype(labels.dtype, np.floating):
        print("Labels appear to be numeric already. Ensuring integer type.")
        return labels.astype(int), cls_to_idx, idx_to_cls

    encoded = np.array([cls_to_idx[l] for l in labels])
    return encoded, cls_to_idx, idx_to_cls

# Apply to Train
y_train_int, class_map, inverse_map = manual_label_encoder(y_train_raw)

# Apply SAME mapping to Test (Critical: Test data must use Train mapping)
try:
    if np.issubdtype(y_test_raw.dtype, np.integer) or np.issubdtype(y_test_raw.dtype, np.floating):
         y_test_int = y_test_raw.astype(int)
    else:
         y_test_int = np.array([class_map[l] for l in y_test_raw])
except KeyError as e:
    print(f"Error: Test set contains a class not seen in Training: {e}")
    raise

print("✓ Label Encoding Complete.")
print("Class Mapping:", class_map)
print(f"Encoded Train Sample: {y_train_raw[0]} -> {y_train_int[0]}")



--- CELL 3: MANUAL LABEL ENCODING ---
Labels appear to be numeric already. Ensuring integer type.
✓ Label Encoding Complete.
Class Mapping: {np.int64(0): 0, np.int64(1): 1, np.int64(2): 2, np.int64(3): 3, np.int64(4): 4, np.int64(5): 5}
Encoded Train Sample: 1 -> 1


In [4]:
# -----------------------------------------------------------
# CELL 4: MANUAL STRATIFIED TRAIN/VAL SPLIT (NO SKLEARN)
# -----------------------------------------------------------
print("\n--- CELL 4: MANUAL STRATIFIED SPLIT ---")

def stratified_split(features, labels, val_ratio=0.2, seed=42):
    """
    Splits data into Train and Validation sets while preserving class percentages.
    Implemented without sklearn.
    """
    np.random.seed(seed)

    train_indices = []
    val_indices = []

    unique_classes = np.unique(labels)

    print(f"Performing Stratified Split (Val Ratio: {val_ratio})...")

    for c in unique_classes:
        # 1. Find all indices that belong to class 'c'
        indices = np.where(labels == c)[0]

        # 2. Shuffle indices manually
        np.random.shuffle(indices)

        # 3. Determine split point
        n_samples = len(indices)
        n_val = int(n_samples * val_ratio)

        # 4. Slice indices
        val_idx = indices[:n_val]
        train_idx = indices[n_val:]

        # 5. Append to master lists
        val_indices.extend(val_idx)
        train_indices.extend(train_idx)

    # 6. Final Shuffle of the combined lists to avoid ordering bias
    np.random.shuffle(train_indices)
    np.random.shuffle(val_indices)

    # 7. Create the actual arrays
    X_train = features[train_indices]
    y_train = labels[train_indices]

    X_val = features[val_indices]
    y_val = labels[val_indices]

    return X_train, y_train, X_val, y_val

# Execute Split
X_train, y_train, X_val, y_val = stratified_split(X_train_raw, y_train_int, val_ratio=0.2)

print("\n--- Split Results ---")
print(f"Training Set:   {X_train.shape} | Labels: {y_train.shape}")
print(f"Validation Set: {X_val.shape}   | Labels: {y_val.shape}")
print(f"Test Set:       {X_test_raw.shape}  | Labels: {y_test_int.shape}")




--- CELL 4: MANUAL STRATIFIED SPLIT ---
Performing Stratified Split (Val Ratio: 0.2)...

--- Split Results ---
Training Set:   (1756, 2048) | Labels: (1756,)
Validation Set: (435, 2048)   | Labels: (435,)
Test Set:       (548, 2048)  | Labels: (548,)


In [5]:
# -----------------------------------------------------------
# CELL 5: DISTRIBUTION CHECK (VISUALIZATION)
# -----------------------------------------------------------
print("\n--- CELL 5: DISTRIBUTION CHECK ---")

def plot_distribution(y_train, y_val, class_map):
    classes = list(class_map.keys())
    n_classes = len(classes)

    # Count per class using numpy (Manual bin counts)
    train_counts = np.bincount(y_train, minlength=n_classes)
    val_counts = np.bincount(y_val, minlength=n_classes)

    # Calculate Percentages
    train_pct = train_counts / len(y_train)
    val_pct = val_counts / len(y_val)

    x = np.arange(n_classes)
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    rects1 = ax.bar(x - width/2, train_pct, width, label='Train %')
    rects2 = ax.bar(x + width/2, val_pct, width, label='Val %')

    ax.set_ylabel('Percentage of Dataset')
    ax.set_title('Class Distribution: Train vs Validation (Stratification Check)')
    ax.set_xticks(x)
    ax.set_xticklabels(classes, rotation=45)
    ax.legend()

    plt.show()

    # Assert checks
    print("Distribution Check:")
    for i, cls in enumerate(classes):
        diff = abs(train_pct[i] - val_pct[i])
        status = "OK" if diff < 0.05 else "WARNING: Imbalanced"
        print(f"  {cls}: Train={train_pct[i]:.2%}, Val={val_pct[i]:.2%} -> {status}")

plot_distribution(y_train, y_val, class_map)


--- CELL 5: DISTRIBUTION CHECK ---


Distribution Check:
  0: Train=3.87%, Val=3.68% -> OK
  1: Train=61.05%, Val=61.61% -> OK
  2: Train=10.48%, Val=10.34% -> OK
  3: Train=5.13%, Val=5.06% -> OK
  4: Train=5.47%, Val=5.29% -> OK
  5: Train=14.01%, Val=14.02% -> OK


In [6]:
# -----------------------------------------------------------
# CELL 6: SAVE FINAL ARRAYS & METADATA
# -----------------------------------------------------------
print("\n--- CELL 6: SAVING DATA ---")
print("Saving processed data to Drive...")

# 1. Save Arrays
np.save(f"{FEATURE_DIR}/final_X_train.npy", X_train)
np.save(f"{FEATURE_DIR}/final_y_train.npy", y_train)
np.save(f"{FEATURE_DIR}/final_X_val.npy", X_val)
np.save(f"{FEATURE_DIR}/final_y_val.npy", y_val)
np.save(f"{FEATURE_DIR}/final_X_test.npy", X_test_raw)
np.save(f"{FEATURE_DIR}/final_y_test.npy", y_test_int)

# 2. Save Metadata (Dictionary mapping) using Pickle
metadata = {
    "class_to_idx": class_map,
    "idx_to_class": inverse_map,
    "num_classes": len(class_map),
    "feature_dim": X_train.shape[1]
}

with open(f"{FEATURE_DIR}/dataset_metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

print("✓ All files saved to:", FEATURE_DIR)


--- CELL 6: SAVING DATA ---
Saving processed data to Drive...
✓ All files saved to: /content/drive/MyDrive/FabricDefect/features


In [7]:
# -----------------------------------------------------------
# CELL 7: HELPER FUNCTIONS (ONE-HOT & METRICS)
# -----------------------------------------------------------
def to_one_hot(y, num_classes):
    """
    Manually converts integer labels to one-hot vectors.
    y: array of shape (N,)
    output: array of shape (N, num_classes)
    """
    N = y.shape[0]
    one_hot = np.zeros((N, num_classes))
    one_hot[np.arange(N), y] = 1
    return one_hot

def get_accuracy(y_pred, y_true):
    """
    Calculates accuracy percentage manually.
    """
    predictions = np.argmax(y_pred, axis=1)
    correct = np.sum(predictions == y_true)
    return correct / len(y_true)

print("✓ Helper functions defined.")


✓ Helper functions defined.


In [8]:
# -----------------------------------------------------------
# CELL 8: SOFTMAX CLASSIFIER CLASS (MANUAL MATH)
# -----------------------------------------------------------
class SoftmaxClassifier_NumPy:
    def __init__(self, input_dim, num_classes, lr=0.01):
        self.lr = lr

        # 1. Initialize Weights (Xavier Initialization for stability)
        # Random values between -limit and +limit
        limit = np.sqrt(6 / (input_dim + num_classes))
        self.W = np.random.uniform(-limit, limit, (input_dim, num_classes))
        self.b = np.zeros((1, num_classes))

        print(f"Model Initialized. Input: {input_dim}, Classes: {num_classes}")

    def softmax(self, z):
        """
        Compute Softmax: exp(z) / sum(exp(z))
        Includes stability fix (subtracting max) to prevent overflow.
        """
        # Keepdims=True is crucial for broadcasting
        z_safe = z - np.max(z, axis=1, keepdims=True)
        exp_z = np.exp(z_safe)
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def forward(self, X):
        """
        Forward Pass: Z = Xw + b
        """
        z = np.dot(X, self.W) + self.b
        return self.softmax(z)

    def compute_loss(self, y_pred_probs, y_true_onehot):
        """
        Cross Entropy Loss: -mean( sum(y_true * log(y_pred)) )
        """
        N = y_pred_probs.shape[0]
        # Clip to prevent log(0) which causes NaN
        y_pred_safe = np.clip(y_pred_probs, 1e-15, 1 - 1e-15)

        loss = -np.sum(y_true_onehot * np.log(y_pred_safe)) / N
        return loss

    def backward(self, X, y_pred_probs, y_true_onehot):
        """
        Backward Pass (Gradient Descent).
        Derivative of CrossEntropy + Softmax is simply (Prediction - Target).
        """
        N = X.shape[0]

        # 1. Calculate Error (dz)
        error = y_pred_probs - y_true_onehot

        # 2. Calculate Gradients (dW, db)
        dw = np.dot(X.T, error) / N
        db = np.sum(error, axis=0, keepdims=True) / N

        # 3. Update Parameters (SGD)
        self.W -= self.lr * dw
        self.b -= self.lr * db

print("✓ Classifier Class defined.")


✓ Classifier Class defined.


In [9]:
# -----------------------------------------------------------
# CELL 9: TRAINING LOOP
# -----------------------------------------------------------
# Hyperparameters
INPUT_DIM = X_train.shape[1] # 2048
NUM_CLASSES = len(metadata['class_to_idx'])
LEARNING_RATE = 0.01
EPOCHS = 1000  # Increase this for better results

# Initialize Model
model = SoftmaxClassifier_NumPy(INPUT_DIM, NUM_CLASSES, LEARNING_RATE)

# History lists for plotting
train_losses = []
val_accuracies = []

print(f"\n--- Starting Training ({EPOCHS} Epochs) ---")

# Convert labels to one-hot once to save time inside loop
y_train_onehot = to_one_hot(y_train, NUM_CLASSES)

for epoch in range(EPOCHS):
    # 1. Forward Pass
    probs = model.forward(X_train)

    # 2. Compute Loss
    loss = model.compute_loss(probs, y_train_onehot)

    # 3. Backward Pass (Update Weights)
    model.backward(X_train, probs, y_train_onehot)

    # 4. Validation & Logging (every 100 epochs)
    if (epoch + 1) % 100 == 0:
        val_probs = model.forward(X_val)
        val_acc = get_accuracy(val_probs, y_val)

        train_losses.append(loss)
        val_accuracies.append(val_acc)

        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {loss:.4f} | Val Acc: {val_acc*100:.2f}%")

print("✓ Training Complete.")

Model Initialized. Input: 2048, Classes: 6

--- Starting Training (1000 Epochs) ---
Epoch 100/1000 | Loss: 0.3588 | Val Acc: 86.44%
Epoch 200/1000 | Loss: 0.2933 | Val Acc: 87.36%
Epoch 300/1000 | Loss: 0.2618 | Val Acc: 88.74%
Epoch 400/1000 | Loss: 0.2409 | Val Acc: 89.43%
Epoch 500/1000 | Loss: 0.2254 | Val Acc: 90.11%
Epoch 600/1000 | Loss: 0.2132 | Val Acc: 91.26%
Epoch 700/1000 | Loss: 0.2033 | Val Acc: 91.26%
Epoch 800/1000 | Loss: 0.1950 | Val Acc: 91.26%
Epoch 900/1000 | Loss: 0.1880 | Val Acc: 91.95%
Epoch 1000/1000 | Loss: 0.1818 | Val Acc: 91.95%
✓ Training Complete.


In [10]:
# -----------------------------------------------------------
# CELL 10: PLOTTING RESULTS
# -----------------------------------------------------------
fig, ax1 = plt.subplots(figsize=(8, 4))

ax1.set_xlabel('Epochs (x100)')
ax1.set_ylabel('Loss', color='tab:red')
ax1.plot(train_losses, color='tab:red', label='Training Loss')
ax1.tick_params(axis='y', labelcolor='tab:red')

ax2 = ax1.twinx()
ax2.set_ylabel('Accuracy', color='tab:blue')
ax2.plot(val_accuracies, color='tab:blue', label='Val Accuracy')
ax2.tick_params(axis='y', labelcolor='tab:blue')

plt.title('Training Loss vs Validation Accuracy')
fig.tight_layout()
plt.show()


In [13]:
# -----------------------------------------------------------
# CELL 11: FINAL EVALUATION ON TEST SET
# -----------------------------------------------------------
print("\n--- FINAL TEST EVALUATION ---")

# 1. Predict on Test Set
test_probs = model.forward(X_test_raw)
test_preds = np.argmax(test_probs, axis=1)

# 2. Calculate Final Accuracy
test_acc = get_accuracy(test_probs, y_test_int)
print(f"🏆 Test Accuracy: {test_acc*100:.2f}%")

# 3. Manual Confusion Matrix (No Sklearn)
print("\n[Confusion Matrix]")
conf_matrix = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)

for true, pred in zip(y_test_int, test_preds):
    conf_matrix[true][pred] += 1

# Pretty Print Matrix - FIXED VERSION
# Get class names properly
if isinstance(metadata['class_to_idx'], dict):
    # If it's a dictionary mapping class names to indices
    classes = list(metadata['class_to_idx'].keys())
    # Format each class name (first 3 chars)
    class_display = [f"{c[:3]:<3}" if isinstance(c, str) else f"{str(c)[:3]:<3}" for c in classes]
else:
    # If it's something else (like an array), create generic labels
    classes = [f"Class_{i}" for i in range(NUM_CLASSES)]
    class_display = [f"{c[:3]:<3}" for c in classes]

print(f"{'True \\ Pred':<15} | " + " | ".join(class_display))
print("-" * 60)

for i, row in enumerate(conf_matrix):
    # Get the class label for this row
    if isinstance(metadata['class_to_idx'], dict):
        # Need to reverse the mapping: index -> class name
        idx_to_class = {v: k for k, v in metadata['class_to_idx'].items()}
        true_label = idx_to_class.get(i, f"Class_{i}")
    else:
        true_label = f"Class_{i}"

    row_str = " | ".join([f"{val:<3}" for val in row])
    print(f"{true_label:<15} | {row_str}")

# 4. Save Trained Model Weights
# Define MODEL_DIR if it doesn't exist
MODEL_DIR = "model_weights"  # Add this line
import os
os.makedirs(MODEL_DIR, exist_ok=True)  # Create directory if it doesn't exist

weights_path = f"{MODEL_DIR}/manual_classifier_weights.npz"
np.savez(weights_path, W=model.W, b=model.b)
print(f"\n✓ Model weights saved to: {weights_path}")


--- FINAL TEST EVALUATION ---
🏆 Test Accuracy: 92.34%

[Confusion Matrix]
True \ Pred     | 0   | 1   | 2   | 3   | 4   | 5  
------------------------------------------------------------
0               | 10  | 0   | 5   | 2   | 0   | 0  
1               | 0   | 322 | 0   | 0   | 0   | 4  
2               | 2   | 1   | 40  | 3   | 6   | 0  
3               | 2   | 0   | 4   | 18  | 0   | 0  
4               | 0   | 1   | 6   | 0   | 31  | 0  
5               | 0   | 6   | 0   | 0   | 0   | 85 

✓ Model weights saved to: model_weights/manual_classifier_weights.npz
